# Replication: There’s Life in the Old GLM Yet!

**Paper**: When a modern foundation model meets decades of actuarial tradition

**Authors**: Cillian Williamson, Scott Hawes, Jin Cui, Karol Gawłowski

This notebook reproduces the exact experiments described in the paper:
- Compare TabPFN (pretrained foundation model) vs. traditional GLM on UK motor lapse prediction
- Benchmark against CatBoost, XGBoost, and RandomForest for completeness
- Evaluate calibration using post-hoc isotonic regression
- Reproduce all reported metrics and figures

**Dataset**: eudirectlapse (23,060 motor insurance policies, 13% lapse rate)

**Reproducibility**: Seed=45, 70:30 stratified train/test split, deterministic backends enabled.

## 1. Install and Pin Dependencies

Install the exact versions used in the paper experiments. Adjust versions if you have different upstream TabPFN versions installed.

In [1]:
import subprocess
import sys

# Core data science and ML packages
packages = [
    'pandas>=1.3.0',
    'numpy>=1.21.0',
    'scikit-learn>=1.0.0',
    'torch>=1.10.0',
    'matplotlib>=3.5.0',
    'seaborn>=0.11.0',
    'catboost>=1.0.0',
    'xgboost>=1.5.0',
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', package])

print("\n✓ All dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 27.2 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tabpfn-extensions 0.1.3 requires pandas<3,>=1.4.0, but you have pandas 3.0.2 which is incompatible.
tabpfn-client 0.2.8 requires pandas<=2.3.3,>=2.1.2, but you have pandas 3.0.2 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


  Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 19.1 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tabpfn-extensions 0.1.3 requires pandas<3,>=1.4.0, but you have pandas 3.0.2 which is incompatible.
tabpfn-extensions 0.1.3 requires scikit-learn<1.7,>=1.2.0, but you have scikit-learn 1.8.0 which is incompatible.
tabpfn-client 0.2.8 requires pandas<=2.3.3,>=2.1.2, but you have pandas 3.0.2 which is incompatible.
tabpfn-client 0.2.8 requires scikit-learn<=1.6.1,>=1.3.1, but you have scikit-learn 1.8.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


...


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip



✓ All dependencies installed successfully.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


## 2. Import Libraries and Project Utilities

In [ ]:
## Setup: Install Optional Dependencies for Dataset Loading

# Optional: If you have R installed with CASdatasets package, install pyreadr for direct loading
# pip install pyreadr

# If R is not available, the notebook will provide instructions to download the CSV manually
# eudirectlapse is a public dataset from the CASdatasets package on CRAN

print("Dataset Loading Strategy:")
print("1. First attempt: Load from local R installation (if R + CASdatasets is installed)")
print("2. Fallback: Manual download instructions provided")
print("\nThe eudirectlapse dataset is public and open-source.")
print("Source: CASdatasets R package (https://cran.r-project.org/web/packages/CASdatasets/)")

In [2]:
# Standard library
import os
import json
from pathlib import Path
from datetime import datetime

# Data manipulation
import numpy as np
import pandas as pd

# Machine learning and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, f1_score, confusion_matrix
from sklearn.calibration import IsotonicRegression

# Tree-based models for comparison
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# PyTorch and TabPFN
import torch
import warnings
warnings.filterwarnings('ignore')

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All imports successful.")

✓ All imports successful.


## 3. Set Global Reproducibility Controls

Enable deterministic behavior across all libraries.

In [3]:
# Paper-consistent seed
SEED = 45
TEST_SIZE = 0.30
RANDOM_STATE = SEED

# Set seeds for reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device selection
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Random Seed: {SEED}")
print(f"Train/Test Split: {1-TEST_SIZE:.1%} / {TEST_SIZE:.1%}")
print(f"Device: {DEVICE}")
print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

Random Seed: 45
Train/Test Split: 70.0% / 30.0%
Device: cpu
PyTorch Version: 2.11.0
NumPy Version: 2.4.4
Pandas Version: 3.0.2


## 4. Resolve Repository Paths and Artifact Locations

Configure paths for datasets and output directories. Adapt these based on your local setup.

In [4]:
# Define output directories (data will be fetched from web)
import io
import urllib.request

# Output directories for results
OUTPUT_DIR = Path.cwd().parent.parent / 'outputs' / 'replication'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TABLES_DIR = OUTPUT_DIR / 'tables'
FIGURES_DIR = OUTPUT_DIR / 'figures'
PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'

for directory in [TABLES_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Output Directory: {OUTPUT_DIR}")
print(f"\n✓ Directory structure initialized.")
print(f"✓ Data will be fetched from web sources (CASdatasets package).")

Output Directory: /Users/Scott/Documents/Data Science/ADSWP/TabPFN-work-scott/outputs/replication

✓ Directory structure initialized.
✓ Data will be fetched from web sources (CASdatasets package).


## 5. Load Benchmark Dataset

Load eudirectlapse and validate schema assumptions from the paper.

In [5]:
# Load data from CASdatasets (public R package via web source)
print("Fetching eudirectlapse dataset from CASdatasets package...")

import os
import subprocess

# Dynamically find the CASdatasets package location via R
RDATA_PATH = None
try:
    r_pkg_path = subprocess.check_output(
        ['Rscript', '-e', "cat(find.package('CASdatasets', quiet=TRUE))"],
        stderr=subprocess.DEVNULL
    ).decode().strip()
    if r_pkg_path:
        for ext in ['eudirectlapse.rda', 'eudirectlapse.RData']:
            candidate = os.path.join(r_pkg_path, 'data', ext)
            if os.path.exists(candidate):
                RDATA_PATH = candidate
                break
except Exception:
    pass

# Fallback: common installation paths
if RDATA_PATH is None:
    FALLBACK_PATHS = [
        '/opt/homebrew/lib/R/4.6/site-library/CASdatasets/data/eudirectlapse.rda',
        '/opt/homebrew/lib/R/4.5/site-library/CASdatasets/data/eudirectlapse.rda',
        '/usr/local/lib/R/site-library/CASdatasets/data/eudirectlapse.rda',
        '/usr/local/lib/R/site-library/CASdatasets/data/eudirectlapse.RData',
        '/usr/lib/R/site-library/CASdatasets/data/eudirectlapse.rda',
        os.path.expanduser('~/R/aarch64-apple-darwin20/4.6/CASdatasets/data/eudirectlapse.rda'),
        os.path.expanduser('~/R/x86_64-pc-linux-gnu-library/4.6/CASdatasets/data/eudirectlapse.rda'),
    ]
    for path in FALLBACK_PATHS:
        if os.path.exists(path):
            RDATA_PATH = path
            break

df = None
if RDATA_PATH:
    try:
        import pyreadr
        print(f"  - Loading from: {RDATA_PATH}")
        result = pyreadr.read_r(RDATA_PATH)
        key = list(result.keys())[0] if result.keys() else None
        df = result[key]
        print("  ✓ Loaded successfully")
    except Exception as e:
        print(f"  ⚠ pyreadr failed: {e}")

if df is None:
    print("\n" + "="*80)
    print("DATASET SETUP REQUIRED - eudirectlapse from CASdatasets")
    print("="*80)
    print("\nThis dataset is public and from the CASdatasets R package.")
    print("\nStep 1: Install R")
    print("  macOS:  brew install r")
    print("  Ubuntu: sudo apt install r-base")
    print("\nStep 2: Install CASdatasets in R")
    print("  Rscript -e \"install.packages('xts', repos='https://cloud.r-project.org')\"")
    print("  Rscript -e \"install.packages('https://cas.uqam.ca/pub/src/contrib/CASdatasets_1.2-0.tar.gz', repos=NULL, type='source')\"")
    print("\nStep 3: Install pyreadr")
    print("  pip install pyreadr")
    print("\nAlternatively — manual CSV export:")
    print("  library(CASdatasets); data(eudirectlapse)")
    print("  write.csv(eudirectlapse, 'eudirectlapse.csv', row.names=FALSE)")
    print("  # then: df = pd.read_csv('eudirectlapse.csv')")
    print("\nSource: https://cas.uqam.ca/")
    print("="*80)
    raise FileNotFoundError("Please install dataset as shown above")

print(f"\nDataset Shape: {df.shape}")
print(f"\nFirst Few Rows:")
print(df.head())
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values:")
print(df.isnull().sum())

expected_rows = 23060
actual_rows = len(df)
print(f"\nDataset Validation:")
print(f"  Expected Rows: {expected_rows:,} | Actual: {actual_rows:,}")

target_candidates = [col for col in df.columns if col.lower() in ['lapse', 'target', 'churn', 'y']]
if target_candidates:
    target_col = target_candidates[0]
    lapse_rate = df[target_col].mean()
    print(f"  Target Column: {target_col}")
    print(f"  Lapse Rate: {lapse_rate:.2%}")
else:
    print("  ⚠ Target column not auto-detected. Columns:")
    print(f"    {list(df.columns)}")
    target_col = None


Fetching eudirectlapse dataset from CASdatasets package...
  - Loading from: /opt/homebrew/lib/R/4.6/site-library/CASdatasets/data/eudirectlapse.rda
  ✓ Loaded successfully

Dataset Shape: (23060, 19)

First Few Rows:
   lapse  polholder_age polholder_BMCevol polholder_diffdriver  \
0      0             38            stable         only partner   
1      1             35            stable                 same   
2      1             29            stable                 same   
3      0             33              down                 same   
4      0             50            stable                 same   

  polholder_gender polholder_job  policy_age              policy_caruse  \
0             Male        normal           1  private or freelance work   
1             Male        normal           1  private or freelance work   
2             Male        normal           0  private or freelance work   
3           Female       medical           2  private or freelance work   
4         

In [6]:
# Data preprocessing: handle categorical features
# Identify feature types
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Use exclude=[np.number] to catch object, category, bool, and any other non-numeric dtypes
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

# Remove target from feature list
if target_col in numeric_cols:
    numeric_cols.remove(target_col)
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print(f"Numeric Features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical Features ({len(categorical_cols)}): {categorical_cols}")

# Encode all non-numeric features
df_encoded = df.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Ensure all feature columns are numeric (catches any remaining edge cases)
feature_cols = [c for c in df_encoded.columns if c != target_col]
df_encoded[feature_cols] = df_encoded[feature_cols].apply(pd.to_numeric, errors='coerce')

print(f"\nDtypes after encoding:")
print(df_encoded.dtypes)
print(f"\n✓ Categorical features encoded. Ready for model training.")


Numeric Features (9): ['polholder_age', 'policy_age', 'policy_nbcontract', 'prem_final', 'prem_last', 'prem_market', 'prem_pure', 'vehicl_age', 'vehicl_agepurchase']
Categorical Features (9): ['polholder_BMCevol', 'polholder_diffdriver', 'polholder_gender', 'polholder_job', 'policy_caruse', 'prem_freqperyear', 'vehicl_garage', 'vehicl_powerkw', 'vehicl_region']

Dtypes after encoding:
lapse                     int32
polholder_age             int32
polholder_BMCevol         int64
polholder_diffdriver      int64
polholder_gender          int64
polholder_job             int64
policy_age                int32
policy_caruse             int64
policy_nbcontract         int32
prem_final              float64
prem_freqperyear          int64
prem_last               float64
prem_market             float64
prem_pure               float64
vehicl_age                int32
vehicl_agepurchase        int32
vehicl_garage             int64
vehicl_powerkw            int64
vehicl_region             int64
dtyp

## 6. Construct Paper-Consistent Splits and Evaluation Protocol

Create stratified 70:30 train/test split as described in the paper.

In [7]:
# Prepare features and target
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

# Stratified train/test split (70:30 with seed=45)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y  # Preserve class distribution
)

print(f"Train Set: {X_train.shape[0]:,} samples ({y_train.mean():.2%} lapse rate)")
print(f"Test Set:  {X_test.shape[0]:,} samples ({y_test.mean():.2%} lapse rate)")
print(f"Total:     {len(df):,} samples")
print(f"\nTrain/Test Distribution:")
print(f"  Train: {(len(X_train)/len(df)):.1%}")
print(f"  Test:  {(len(X_test)/len(df)):.1%}")

# Scale features for models that benefit from it (GLM, neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Data splits created and features scaled.")

Train Set: 16,142 samples (12.81% lapse rate)
Test Set:  6,918 samples (12.81% lapse rate)
Total:     23,060 samples

Train/Test Distribution:
  Train: 70.0%
  Test:  30.0%

✓ Data splits created and features scaled.


In [1]:
# Define evaluation metrics (from paper)
def evaluate_model(y_true, y_pred_proba, y_pred_binary=None, model_name="Model"):
    """
    Evaluate classification model on paper's metrics:
    - ROC AUC: discrimination (ranking ability)
    - Brier Score: calibration (probability accuracy)
    - F1 Score: threshold-based balance on imbalanced data
    """
    results = {}
    
    # ROC AUC (discrimination)
    auc = roc_auc_score(y_true, y_pred_proba)
    results['ROC_AUC'] = auc
    
    # Brier Score (calibration - lower is better)
    brier = brier_score_loss(y_true, y_pred_proba)
    results['Brier'] = brier
    
    # F1 Score (if binary predictions provided)
    if y_pred_binary is None:
        y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    f1 = f1_score(y_true, y_pred_binary)
    results['F1'] = f1
    
    return results

# Initialize results storage
all_results = []

print("✓ Evaluation metrics defined.")
print("  - ROC AUC: discrimination (higher is better, 0.5=random, 1.0=perfect)")
print("  - Brier: calibration (lower is better, 0=perfect, 0.25=worst)")
print("  - F1: threshold-based balance (higher is better)")

✓ Evaluation metrics defined.
  - ROC AUC: discrimination (higher is better, 0.5=random, 1.0=perfect)
  - Brier: calibration (lower is better, 0=perfect, 0.25=worst)
  - F1: threshold-based balance (higher is better)


## 7. Run Pretrained TabPFN Evaluation

Note: TabPFN v2.0 can be run via API or locally. This example shows the local inference approach.

In [ ]:
try:
    from tabpfn_client import set_access_token, TabPFNClassifier
    print("✓ tabpfn-client imported successfully.")
    TABPFN_AVAILABLE = True
except ImportError:
    print("⚠ tabpfn-client not installed. Install via:")
    print("  pip install tabpfn-client")
    TABPFN_AVAILABLE = False

if TABPFN_AVAILABLE:
    # Load API key from .env file (never hardcode credentials in notebooks).
    # Create a .env file in the project root with:
    #   TABPFN_API_KEY=your_key_here
    # Get your key at: https://ux.priorlabs.ai/api/keys
    # (.env is in .gitignore and will not be committed)
    import os
    from dotenv import load_dotenv
    load_dotenv()

    api_key = os.environ.get("TABPFN_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "TABPFN_API_KEY not found. "
            "Create a .env file in the project root with:\n"
            "  TABPFN_API_KEY=your_key_here\n"
            "Get your key at: https://ux.priorlabs.ai/api/keys"
        )

    set_access_token(api_key)
    print("✓ TabPFN API authenticated.")

    print("\nRunning TabPFN inference via API (no row limit, GPU-backed)...")

    # Use the full training set — matches the paper's setup via the PriorLabs API
    clf_tabpfn = TabPFNClassifier()
    clf_tabpfn.fit(X_train, y_train)

    y_pred_proba_tabpfn = clf_tabpfn.predict_proba(X_test)[:, 1]

    # Evaluate
    results_tabpfn = evaluate_model(y_test, y_pred_proba_tabpfn, model_name="TabPFN")
    results_tabpfn['Model'] = 'TabPFN (Raw)'
    all_results.append(results_tabpfn)

    print(f"\n  Results (TabPFN Raw):")
    print(f"    ROC AUC: {results_tabpfn['ROC_AUC']:.4f}")
    print(f"    Brier:   {results_tabpfn['Brier']:.4f}")
    print(f"    F1:      {results_tabpfn['F1']:.4f}")
else:
    print("\nSkipping TabPFN evaluation.")
    y_pred_proba_tabpfn = None


: 

## 8. Probability Calibration: Post-Hoc Isotonic Regression

Apply the calibration technique described in the paper to improve probability accuracy.

In [11]:
if TABPFN_AVAILABLE and y_pred_proba_tabpfn is not None:
    print("Applying post-hoc isotonic calibration...")
    
    # Split test set into calibration and validation
    # Use a portion of test data to learn calibration mapping
    n_test = len(X_test)
    n_calib = n_test // 2
    
    # Calibration set
    y_calib = y_test.iloc[:n_calib].values
    y_pred_proba_calib = y_pred_proba_tabpfn[:n_calib]
    
    # Validation set (for final evaluation)
    y_val = y_test.iloc[n_calib:].values
    y_pred_proba_val = y_pred_proba_tabpfn[n_calib:]
    
    # Fit isotonic calibration on calibration set
    iso_cal = IsotonicRegression(out_of_bounds='clip')
    iso_cal.fit(y_pred_proba_calib, y_calib)
    
    # Apply to validation set
    y_pred_proba_calibrated_val = iso_cal.predict(y_pred_proba_val)
    
    # Evaluate calibrated predictions
    results_tabpfn_cal = evaluate_model(y_val, y_pred_proba_calibrated_val, model_name="TabPFN (Calibrated)")
    results_tabpfn_cal['Model'] = 'TabPFN (Calibrated)'
    all_results.append(results_tabpfn_cal)
    
    print(f"\n  Results (TabPFN Calibrated):")
    print(f"    ROC AUC: {results_tabpfn_cal['ROC_AUC']:.4f}")
    print(f"    Brier:   {results_tabpfn_cal['Brier']:.4f} (improvement: {(results_tabpfn['Brier'] - results_tabpfn_cal['Brier'])*100:.2f}%)")
    print(f"    F1:      {results_tabpfn_cal['F1']:.4f}")
    
    print(f"\n  ✓ Post-hoc calibration complete.")
else:
    print("Skipping calibration (TabPFN predictions not available).")

Applying post-hoc isotonic calibration...

  Results (TabPFN Calibrated):
    ROC AUC: 0.5988
    Brier:   0.1117 (improvement: -0.13%)
    F1:      0.0000

  ✓ Post-hoc calibration complete.


## 9. Run Baseline Models for Comparison

Train GLM, CatBoost, XGBoost, and RandomForest under identical conditions.

In [12]:
# 1. GLM (Logistic Regression)
print("Training GLM (Logistic Regression)...")
clf_glm = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='lbfgs',
    n_jobs=-1
)
clf_glm.fit(X_train_scaled, y_train)
y_pred_proba_glm = clf_glm.predict_proba(X_test_scaled)[:, 1]
results_glm = evaluate_model(y_test, y_pred_proba_glm, model_name="GLM")
results_glm['Model'] = 'GLM'
all_results.append(results_glm)

print(f"  ROC AUC: {results_glm['ROC_AUC']:.4f}")
print(f"  Brier:   {results_glm['Brier']:.4f}")
print(f"  F1:      {results_glm['F1']:.4f}")

Training GLM (Logistic Regression)...
  ROC AUC: 0.6205
  Brier:   0.1092
  F1:      0.0000


In [13]:
# 2. CatBoost
print("Training CatBoost...")
clf_catboost = CatBoostClassifier(
    iterations=100,
    random_state=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False
)
clf_catboost.fit(X_train, y_train, verbose=False)
y_pred_proba_catboost = clf_catboost.predict_proba(X_test)[:, 1]
results_catboost = evaluate_model(y_test, y_pred_proba_catboost, model_name="CatBoost")
results_catboost['Model'] = 'CatBoost'
all_results.append(results_catboost)

print(f"  ROC AUC: {results_catboost['ROC_AUC']:.4f}")
print(f"  Brier:   {results_catboost['Brier']:.4f}")
print(f"  F1:      {results_catboost['F1']:.4f}")

Training CatBoost...
  ROC AUC: 0.6216
  Brier:   0.1088
  F1:      0.0000


In [1]:
# 3. XGBoost
print("Training XGBoost...")
clf_xgb = XGBClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    verbosity=0,
    n_jobs=1,  # -1 can crash on Apple Silicon
)
clf_xgb.fit(X_train, y_train, verbose=False)
y_pred_proba_xgb = clf_xgb.predict_proba(X_test)[:, 1]
results_xgb = evaluate_model(y_test, y_pred_proba_xgb, model_name="XGBoost")
results_xgb['Model'] = 'XGBoost'
all_results.append(results_xgb)

print(f"  ROC AUC: {results_xgb['ROC_AUC']:.4f}")
print(f"  Brier:   {results_xgb['Brier']:.4f}")
print(f"  F1:      {results_xgb['F1']:.4f}")


Training XGBoost...


NameError: name 'XGBClassifier' is not defined

In [ ]:
# 4. RandomForest
print("Training RandomForest...")
clf_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=1,  # -1 can crash on Apple Silicon
)
clf_rf.fit(X_train, y_train)
y_pred_proba_rf = clf_rf.predict_proba(X_test)[:, 1]
results_rf = evaluate_model(y_test, y_pred_proba_rf, model_name="RandomForest")
results_rf['Model'] = 'RandomForest'
all_results.append(results_rf)

print(f"  ROC AUC: {results_rf['ROC_AUC']:.4f}")
print(f"  Brier:   {results_rf['Brier']:.4f}")
print(f"  F1:      {results_rf['F1']:.4f}")


## 10. Aggregate Metrics Across All Models

In [ ]:
# Consolidate results into a single DataFrame
results_df = pd.DataFrame(all_results)

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS TABLE")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

# Sort by ROC AUC (discrimination) for easy comparison
results_sorted_auc = results_df.sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
print("\nRanked by ROC AUC (Discrimination):")
print(results_sorted_auc[['Model', 'ROC_AUC']].to_string(index=False))

# Sort by Brier (calibration) for easy comparison
results_sorted_brier = results_df.sort_values('Brier', ascending=True).reset_index(drop=True)
print("\nRanked by Brier Score (Calibration, lower is better):")
print(results_sorted_brier[['Model', 'Brier']].to_string(index=False))

## 11. Recreate Paper Tables

Generate publication-style summary table matching the paper's presentation.

In [ ]:
# Create a styled comparison table
paper_table = results_df[['Model', 'ROC_AUC', 'Brier', 'F1']].copy()
paper_table = paper_table.round(4)

# Add interpretability and speed information from paper
interpretability = {
    'GLM': 'Coefficients',
    'TabPFN (Raw)': 'Black box',
    'TabPFN (Calibrated)': 'Black box',
    'CatBoost': 'Black box',
    'XGBoost': 'Black box',
    'RandomForest': 'Black box'
}

inference_speed = {
    'GLM': '0.01s',
    'TabPFN (Raw)': '7.6s (API)',
    'TabPFN (Calibrated)': '7.6s (API)',
    'CatBoost': '~0.1s',
    'XGBoost': '~0.05s',
    'RandomForest': '~0.1s'
}

paper_table['Interpretability'] = paper_table['Model'].map(interpretability)
paper_table['Inference Speed'] = paper_table['Model'].map(inference_speed)

print("\nPUBLICATION-STYLE COMPARISON TABLE:")
print(paper_table.to_string(index=False))

# Save to CSV
paper_table.to_csv(TABLES_DIR / 'paper_comparison_table.csv', index=False)
print(f"\n✓ Saved to: {TABLES_DIR / 'paper_comparison_table.csv'}")

## 12. Recreate Paper Figures

Generate key visualizations from the paper.

In [ ]:
# Figure 1: ROC AUC Comparison
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
results_sorted = results_df.sort_values('ROC_AUC', ascending=True)
colors = ['#d62728' if 'TabPFN' in model else '#1f77b4' if model == 'GLM' else '#ff7f0e' 
          for model in results_sorted['Model']]
ax.barh(results_sorted['Model'], results_sorted['ROC_AUC'], color=colors)
ax.set_xlabel('ROC AUC Score', fontsize=12, fontweight='bold')
ax.set_title('Model Discrimination Comparison (ROC AUC)', fontsize=14, fontweight='bold')
ax.set_xlim([0.55, 0.62])
for i, v in enumerate(results_sorted['ROC_AUC']):
    ax.text(v - 0.005, i, f'{v:.4f}', va='center', ha='right', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_roc_auc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {FIGURES_DIR / 'fig_roc_auc_comparison.png'}")

In [ ]:
# Figure 2: Brier Score Comparison (Calibration)
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
results_sorted_brier = results_df.sort_values('Brier', ascending=False)
colors_brier = ['#d62728' if 'TabPFN' in model else '#1f77b4' if model == 'GLM' else '#ff7f0e' 
                for model in results_sorted_brier['Model']]
ax.barh(results_sorted_brier['Model'], results_sorted_brier['Brier'], color=colors_brier)
ax.set_xlabel('Brier Score (lower is better)', fontsize=12, fontweight='bold')
ax.set_title('Model Calibration Comparison (Brier Score)', fontsize=14, fontweight='bold')
for i, v in enumerate(results_sorted_brier['Brier']):
    ax.text(v + 0.0005, i, f'{v:.4f}', va='center', ha='left', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_brier_score_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {FIGURES_DIR / 'fig_brier_score_comparison.png'}")

In [ ]:
# Figure 3: Multi-metric Heatmap
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
heatmap_data = results_df[['Model', 'ROC_AUC', 'Brier', 'F1']].set_index('Model')
sns.heatmap(heatmap_data.T, annot=True, fmt='.4f', cmap='RdYlGn', cbar_kws={'label': 'Score'}, ax=ax)
ax.set_title('All Metrics Heatmap (Green=Better for ROC AUC and F1, Red=Better for Brier)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_all_metrics_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {FIGURES_DIR / 'fig_all_metrics_heatmap.png'}")

## 13. Export Predictions, Metrics, and Notebook Artifacts

In [ ]:
# Save predictions for further analysis
if y_pred_proba_tabpfn is not None:
    predictions_df = pd.DataFrame({
        'y_true': y_test.values,
        'TabPFN': y_pred_proba_tabpfn,
        'GLM': y_pred_proba_glm,
        'CatBoost': y_pred_proba_catboost,
        'XGBoost': y_pred_proba_xgb,
        'RandomForest': y_pred_proba_rf,
    })
    predictions_df.to_csv(PREDICTIONS_DIR / 'all_model_predictions.csv', index=False)
    print(f"✓ Predictions saved: {PREDICTIONS_DIR / 'all_model_predictions.csv'}")

# Save results metrics
results_df.to_csv(TABLES_DIR / 'model_metrics.csv', index=False)
print(f"✓ Metrics saved: {TABLES_DIR / 'model_metrics.csv'}")

# Save configuration snapshot
config_snapshot = {
    'Notebook': 'REPLICATION_There_Is_Life_in_the_Old_GLM_Yet',
    'Dataset': 'eudirectlapse',
    'Paper_Title': "There's Life in the Old GLM Yet!",
    'Authors': 'Cillian Williamson, Scott Hawes, Jin Cui, Karol Gawłowski',
    'Seed': SEED,
    'Train_Test_Split': f'{1-TEST_SIZE:.1%}/{TEST_SIZE:.1%}',
    'Stratified': True,
    'Samples_Train': len(X_train),
    'Samples_Test': len(X_test),
    'Features': X.shape[1],
    'Target_Lapse_Rate': float(y.mean()),
    'Device': DEVICE,
    'Timestamp': datetime.now().isoformat(),
}

with open(OUTPUT_DIR / 'replication_config.json', 'w') as f:
    json.dump(config_snapshot, f, indent=2)
print(f"✓ Config saved: {OUTPUT_DIR / 'replication_config.json'}")

print(f"\n{'='*80}")
print("REPLICATION COMPLETE")
print(f"{'='*80}")
print(f"All outputs saved to: {OUTPUT_DIR}")
print(f"  - Tables:       {TABLES_DIR}")
print(f"  - Figures:      {FIGURES_DIR}")
print(f"  - Predictions:  {PREDICTIONS_DIR}")

## Replication Summary

### What We Reproduced:

1. **Dataset**: eudirectlapse (23,060 UK motor insurance policies, 13% lapse rate)
2. **Split**: Stratified 70:30 train/test with seed=45
3. **Models Compared**:
   - TabPFN (pretrained foundation model, no tuning required)
   - GLM (traditional logistic regression)
   - CatBoost, XGBoost, RandomForest (for context)
4. **Metrics**:
   - ROC AUC: discrimination/ranking ability
   - Brier Score: probability calibration accuracy
   - F1: threshold-based balance on imbalanced data
5. **Calibration**: Post-hoc isotonic regression on TabPFN predictions

### Key Finding from the Paper:

On well-structured problems with primarily additive risk factors (like this lapse dataset), **GLM is competitive** with modern foundation models. However, **post-hoc calibration improves TabPFN's probability accuracy** by ~0.87%, translating to real value in pricing and reserving.

### Expected Results:

| Model | ROC AUC | Brier | F1 |
|-------|---------|-------|----|
| GLM | ~0.599 | ~0.1098 | ~0.XX |
| TabPFN (Raw) | ~0.593 | ~0.1108 | ~0.XX |
| TabPFN (Calibrated) | ~0.593 | ~0.1080 ✓ | ~0.XX |
| CatBoost | ~0.591 | - | - |
| XGBoost | ~0.551 | - | - |
| RandomForest | ~0.578 | - | - |

### For Practitioners:

✓ Use GLM when: real-time predictions matter, simple structure, interpretability critical  
✓ Use TabPFN when: probability accuracy matters, workflow simplicity desired, minimal feature engineering needed  
✓ Always test on your specific data before choosing